In [ ]:
import sys
sys.path.insert(0, '../../stock_factor_lab_2025/')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 台股實驗

## 回測期間設定

In [ ]:
START_DATE = '2003-3-31'
END_DATE = '2024-12-31'

## 前置作業

### import

In [ ]:
from get_data import Data
import backtest
from combinations import sim_conditions
import random
import talib
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import itertools

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from itertools import cycle
from plotly.subplots import make_subplots
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
import seaborn as sns
import re

from matplotlib import rcParams
rcParams['font.sans-serif'] = ['Microsoft JhengHei']
mpl.rcParams['axes.unicode_minus'] = False

from dataframe import CustomDataFrame

### get data

In [ ]:
data=Data()

## 資料下載

In [ ]:
profit = data.get('annual_report_fundamentals:常續性稅後淨利')
income_bf_tax = data.get('annual_report_fundamentals:稅前淨利')

# 本益比
pe = data.get('quarter_report:PE')
daily_pe = data.get('price:daily_pe')

roe = data.get('annual_report:ROE')

payout_ratio = data.get('annual_report_fundamentals:股利支付率')

stock_hold = data.get('annual_report_fundamentals:董監持股%')

In [ ]:
long_term_items = [
    '透過損益按公允價值衡量之金融資產－非流動',
    '透過其他綜合損益按公允價值衡量之金融資產－非流動',
    '按攤銷後成本衡量之金融資產－非流動',
    '避險之金融資產－非流動',
    '合約資產－非流動',
    '採權益法之長期股權投資',
    '預付投資款',
    '投資性不動產淨額'
]
# 長期投資項目 (8個)
long_term_data = [data.get(f'annual_report_fundamentals:{item}').fillna(0) for item in long_term_items]
long_term_investment = sum(long_term_data)
# 固定資產
fixed_assets_year = data.get('annual_report_fundamentals:不動產廠房及設備').fillna(0)
# 計算盈再率分子
long_term_investment_df = (long_term_investment - long_term_investment.shift(4))
fixed_assets_df = (fixed_assets_year - fixed_assets_year.shift(4))
# 分母
profit_rol_df = profit.rolling(4).sum()

In [ ]:
orig_rr = (long_term_investment_df + fixed_assets_df) / profit_rol_df
rr = orig_rr[(profit > 0) & (profit_rol_df > 0)] # ['2006':]

In [ ]:
close = data.get('price:close')

comp_profile = data.get('company_profile')
list_stock_data = {}
for index, row in comp_profile.iterrows():
    stock_code = row['company_symbol']
    listed_date = row['ipo_date']
    end_date = listed_date + pd.DateOffset(years=2)
    
    # 創建一個全為 True 的 series
    series = pd.Series(True, index=close.index)
    # 在上市日之前和之後兩年內設置為 False
    series.loc[:end_date] = False
    list_stock_data[stock_code] = series

listed = pd.concat(list_stock_data, axis=1)
listed = CustomDataFrame(listed)

## 原始條件

In [ ]:
roe_rol = roe.rolling(5).mean()
roe_15 = roe_rol > 15

rr_cond = rr < 0.4

payout_ratio_cond = (payout_ratio.rolling(3).min() >= 40)

profit_cond = profit > 500000 # TEJ 的淨利單位是千元

hold_cond = stock_hold > 10

listed = listed.resample('M').last()

In [ ]:
# 每季本益比
pe_entry = pe < 12
pe_exit = pe > 30

# 每月本益比
# 每日本益比resample成每月

pe_cond_entry_daily = (daily_pe < 12).resample('M').last()
pe_cond_exit_daily = (daily_pe > 30).resample('M').last()

---

## 回測

In [ ]:
# 原始條件
orig_all_cond = (roe_15 & rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed)[START_DATE:END_DATE]
# 原始條件 + 每季本益比
orig_all_cond_and_pe = ((orig_all_cond & pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_exit[START_DATE:END_DATE]))
# 原始條件 + 每月本益比
orig_all_cond_and_pe_daily = ((orig_all_cond & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_cond_exit_daily[START_DATE:END_DATE]))

In [ ]:
rep_all_cond_dic = {}

rep_all_cond_dic['台股_原始條件_不含本益比進出場'] = orig_all_cond
rep_all_cond_dic['台股_原始條件_每季本益比進出場'] = orig_all_cond_and_pe
rep_all_cond_dic['台股_原始條件_每月月底本益比進出場'] = orig_all_cond_and_pe_daily

In [ ]:
rep_all_cond = sim_conditions(rep_all_cond_dic, resample='M', data=data)

In [ ]:
fig = rep_all_cond.reports['台股_原始條件_不含本益比進出場'].create_stacked_returns_plot(5)
fig.write_image('./img/台股原始策略不含進出場條件逐年前五高個股獲利貢獻圖.svg')

In [ ]:
# rep_all_cond.plot_creturns()

In [ ]:
rep_all_cond.selected_stock_count_analysis()

In [ ]:
rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].create_stacked_returns_plot(5)

In [ ]:
rep_all_cond.reports['台股_原始條件_不含本益比進出場'].create_stacked_returns_plot(5)

In [ ]:
fig = rep_all_cond.plot_reps_stock_counts()
# fig.savefig('圖 5台股原始策略入選股數變化圖.svg', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].display()

### 切分時間段2003~2009、2009~2024

In [ ]:
orig_all_cond_2003_2009 = (roe_15 & rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed)[START_DATE:'2009-3-31']
orig_all_cond_2009_2024 = (roe_15 & rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed)['2009-3-31':END_DATE]

# orig_all_cond_and_pe_2003_2009 = ((orig_all_cond_2003_2009 & pe_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | pe_exit[START_DATE:'2009-3-31']))
# orig_all_cond_and_pe_2009_2024 = ((orig_all_cond_2009_2024 & pe_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | pe_exit['2009-3-31':END_DATE]))

orig_all_cond_and_pe_daily_2003_2009 = ((orig_all_cond_2003_2009 & pe_cond_entry_daily[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | pe_cond_exit_daily[START_DATE:'2009-3-31']))
orig_all_cond_and_pe_daily_2009_2024 = ((orig_all_cond_2009_2024 & pe_cond_entry_daily['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | pe_cond_exit_daily['2009-3-31':END_DATE]))

In [ ]:
time_period_dic = {}

time_period_dic['台股_原始條件_不含本益比進出場_2003-2009'] = orig_all_cond_2003_2009
time_period_dic['台股_原始條件_不含本益比進出場_2009-2024'] = orig_all_cond_2009_2024


# time_period_dic['台股_原始條件_每季本益比進出場_2003-2009'] = orig_all_cond_and_pe_2003_2009
# time_period_dic['台股_原始條件_每季本益比進出場_2009-2024'] = orig_all_cond_and_pe_2009_2024

time_period_dic['台股_原始條件_每月月底本益比進出場_2003-2009'] = orig_all_cond_and_pe_daily_2003_2009
time_period_dic['台股_原始條件_每月月底本益比進出場_2009-2024'] = orig_all_cond_and_pe_daily_2009_2024

time_period_rep_collec = sim_conditions(time_period_dic, resample="M", data=data)

In [ ]:
time_period_rep_collec.selected_stock_count_analysis()

In [ ]:
time_period_rep_collec.reports['台股_原始條件_每月月底本益比進出場_2003-2009'].plot_strategy_cumm_return(title='台股_原始條件_每月月底本益比進出場_2003-2009_累積報酬')

In [ ]:
rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].plot_strategy_cumm_return(title='台股_原始條件_每月月底本益比進出場_2003-2024_累積報酬')

## 獲利分佈

In [ ]:
orig_strat_rep_test = rep_all_cond.reports['台股_原始條件_每月月底本益比進出場']

pos_df = orig_strat_rep_test.position
close_price_df = close[START_DATE:END_DATE]

In [ ]:
monthly_first = close_price_df.resample('M').first()
monthly_last = close_price_df.resample('M').last()

# 計算股價每月的增長率：(月底股價/月初股價) - 1
monthly_returns_df = (monthly_last / monthly_first) - 1

In [ ]:
# 每月資金分配比例 * 每月股價增長率
returns_pos_df = monthly_returns_df * pos_df.shift() # 向下移動一個row

In [ ]:
returns_pos_df = returns_pos_df.resample('M').last()

In [ ]:
# 轉換成年度數據，將各股在同一年的每月報酬率相加
returns_pos_df_annual = returns_pos_df.resample('A').sum()
# returns_pos_df_annual.sum(axis=1)

In [ ]:
# 畫圖
def create_stacked_returns_plot(returns_pos_df_annual, top_N=None):
    from itertools import cycle

    # 年份列表
    years = returns_pos_df_annual.index.year.tolist()

    # 為每支股票生成唯一的顏色
    stocks = returns_pos_df_annual.columns.tolist()
    n_stocks = len(stocks)

    # 使用 plotly 的顏色配色方案
    all_colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set1 + px.colors.qualitative.Pastel + px.colors.qualitative.Dark2
    color_cycle = cycle(all_colors)
    color_dict = {stock: next(color_cycle) for stock in stocks}

    # 創建用於正收益和負收益的空列表
    positive_traces = []
    negative_traces = []

    # 針對df中每支個股進行處理
    for stock in returns_pos_df_annual.columns:
        stock_returns = returns_pos_df_annual[stock]

        # 分離正收益和負收益
        positive_returns = stock_returns.copy()
        positive_returns[positive_returns <= 0] = 0

        negative_returns = stock_returns.copy()
        negative_returns[negative_returns >= 0] = 0

        # 如果有正收益，創建正收益trace
        if positive_returns.max() > 0:
            for year_idx, value in enumerate(positive_returns):
                if value > 0:  # 只添加正值
                    positive_traces.append({
                        'stock': stock,
                        'year_idx': year_idx,
                        'year': years[year_idx],
                        'return': value,
                        'color': color_dict[stock]
                    })

        # 如果有負收益，創建負收益trace
        if negative_returns.min() < 0:
            for year_idx, value in enumerate(negative_returns):
                if value < 0:  # 只添加負值
                    negative_traces.append({
                        'stock': stock,
                        'year_idx': year_idx,
                        'year': years[year_idx],
                        'return': value,
                        'color': color_dict[stock]
                    })

    # 創建圖表
    fig = go.Figure()

    # 按年份分組並排序正收益
    for year in years:
        year_traces = [t for t in positive_traces if t['year'] == year]
        # 按報酬值從大到小排序
        year_traces.sort(key=lambda x: x['return'], reverse=True)

        if top_N is not None and len(year_traces) > top_N:
            # 分離前N名和其他
            top_traces = year_traces[:top_N]
            other_traces = year_traces[top_N:]

            # 計算"其他"區塊的正報酬的總和
            other_sum = sum(trace['return'] for trace in other_traces)

            # 繪製"其他"區塊在正報酬bar stack的最底下
            if other_sum > 0:
                fig.add_trace(go.Bar(
                    name=f"其他 ({year})",
                    x=[year],
                    y=[other_sum],
                    base=[0],
                    marker=dict(
                        color='white',
                        pattern_shape="/",
                    ),
                    showlegend=False,
                    hovertemplate=f'%{{x}}<br>其他: %{{y:.2%}}<extra></extra>'
                ))

            # 再繪製前N名，堆疊在`其他`區塊的上方
            cumulative = other_sum
            for trace in reversed(top_traces):
                fig.add_trace(go.Bar(
                    name=trace['stock'],
                    x=[year],
                    y=[trace['return']],
                    base=[cumulative],
                    marker_color=trace['color'],
                    showlegend=False,
                    hovertemplate=f'{trace["stock"]}: {trace["return"]:.2%}<extra></extra>'
                ))
                cumulative += trace['return']
        else:
            # 全部顯示（從底部開始堆疊）
            cumulative = 0
            for trace in reversed(year_traces):
                fig.add_trace(go.Bar(
                    name=trace['stock'],
                    x=[year],
                    y=[trace['return']],
                    base=[cumulative],
                    marker_color=trace['color'],
                    showlegend=False,
                    hovertemplate=f'{trace["stock"]}: {trace["return"]:.2%}<extra></extra>'
                ))
                cumulative += trace['return']

    # 按年份分組並排序負收益
    for year in years:
        year_traces = [t for t in negative_traces if t['year'] == year]
        # 按報酬值從大到小排序（負值從小到大）
        year_traces.sort(key=lambda x: x['return'], reverse=True)

        cumulative = 0
        for trace in year_traces:
            fig.add_trace(go.Bar(
                name=trace['stock'],
                x=[year],
                y=[trace['return']],
                base=[cumulative],
                marker_color=trace['color'],
                showlegend=False,
                hovertemplate=f'{trace["stock"]}: {trace["return"]:.2%}<extra></extra>'
            ))
            cumulative += trace['return']

    # 添加唯一的圖例條目
    unique_stocks = sorted(set(trace['stock'] for trace in positive_traces + negative_traces))
    for stock in unique_stocks:
        fig.add_trace(go.Bar(
            name=stock,
            x=[None],
            y=[None],
            marker_color=color_dict[stock],
            showlegend=True,
            hoverinfo='skip'
        ))

    fig.add_trace(go.Bar(
        name="其他",
        x=[None],
        y=[None],
        marker=dict(
            color='white',
            pattern_shape="/",
        ),
        showlegend=True,
        hoverinfo='skip'
    ))

    fig.update_layout(
        barmode='relative',
        title='個別股票逐年獲利貢獻圖',
        xaxis_title='年份',
        yaxis_title='報酬',
        yaxis_tickformat=".0%",
        xaxis=dict(
            tickmode='array',
            tickvals=list(range(min(years), max(years) + 1)),
            ticktext=list(range(min(years), max(years) + 1))
        ),
        showlegend=True,
        legend_title='股票',
        hovermode='closest',
        width=1400,  # 調整寬度以顯示所有年份
        height=600,
    )

    # 添加水平線
    fig.add_hline(y=0, line_width=1, line_dash="solid", line_color="black")

    return fig

In [ ]:
fig = create_stacked_returns_plot(returns_pos_df_annual, 5)
# fig.write_html('./img/圖 6 台股原始策略加上進出場條件逐年前五高個股獲利貢獻圖.html')
fig.write_image('./img/圖 6 台股原始策略加上進出場條件逐年前五高個股獲利貢獻圖.svg')

---

## 台股_布林通道濾網

In [ ]:
# 大盤股價
taiex_close = data.get('taiex:close')['2000-3':END_DATE]
# 布林通道上中下通道
upperband, middleband, lowerband = talib.BBANDS(taiex_close.close, timeperiod=300, nbdevup=2.0, nbdevdn=2.0)

### 建立濾網

In [ ]:
tw_bollinger_signal = pd.Series(True, index=taiex_close.index)

first_break_price = None
crossed_above_lower = False
in_selling_state = False

for date in taiex_close.index:
    price = taiex_close.close[date]
    lower = lowerband[date]
    middle = middleband[date]
    
    if price < lower and first_break_price is None:
        first_break_price = price
        crossed_above_lower = False
    
    elif price > lower:
        crossed_above_lower = True
    
    if price < lower and crossed_above_lower and price < first_break_price:
        
        tw_bollinger_signal[date] = False
        in_selling_state = True
    
    if price > middle:
        tw_bollinger_signal[date] = True
        first_break_price = None
        crossed_above_lower = False
        in_selling_state = False
    
    if in_selling_state and price <= middle:
        tw_bollinger_signal[date] = False

### 台股布林通道濾網_視覺化

In [ ]:
plt.figure(figsize=(20, 6))

plt.plot(upperband[START_DATE:END_DATE], label="upperband", color='r', linestyle='solid', linewidth=1)
plt.plot(middleband[START_DATE:END_DATE], label="middleband", color='g', linestyle='--', linewidth=1)
plt.plot(lowerband[START_DATE:END_DATE], label="lowerband", color='b', linestyle='solid', linewidth=1)
plt.plot(taiex_close[START_DATE:END_DATE], label="台股大盤", color='black', linewidth=0.8)

spans = [
    ('2008-7-25', '2009-5-5'),
    ('2011-9-14', '2012-3-1'),
    ('2015-9-10', '2016-6-6'),
    ('2018-12-6', '2019-3-19'),
    ('2022-6-20', '2023-3-3')
]
# 繪製灰底和添加標籤
for start, end in spans:
    # 添加灰底
    plt.axvspan(pd.Timestamp(start), pd.Timestamp(end), color='gray', alpha=0.25)
    
    mid_point = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start))/2
    
    label_text = f'{start}~{end}'
    plt.text(mid_point, plt.ylim()[1]*1.05, label_text,
             horizontalalignment='center',
             fontsize=12)

plt.title("台股大盤布林通道（MA300 標準差=2）", fontsize=16, pad=40)
plt.xlabel("年份", fontsize=14)
plt.xticks(fontsize=14)
plt.ylabel("台股大盤")
plt.yticks(fontsize=14)

plt.legend(fontsize=14)
plt.grid(True, linestyle='--', alpha=0.4)

plt.margins(y=0.1)

plt.show()

### 範例圖

In [ ]:
plt.figure(figsize=(20, 6))

# 2008-6-30第一次跌破下通道
# pd.Timestamp('2008-7-25'), pd.Timestamp('2009-5-5')

plt.plot(upperband['2008-6-1':'2009-5-15'], label="upperband", color='r', linestyle='solid', linewidth=1)
plt.plot(middleband['2008-6-1':'2009-5-15'], label="middleband", color='g', linestyle='--', linewidth=1)
plt.plot(lowerband['2008-6-1':'2009-5-15'], label="lowerband", color='b', linestyle='solid', linewidth=1)
plt.plot(taiex_close['2008-6-1':'2009-5-15'], label="台股大盤", color='black', linewidth=0.8)

plt.axvspan(pd.Timestamp('2008-7-25'), pd.Timestamp('2009-5-5'), color='gray', alpha=0.25)

# 箭頭
plt.annotate('收盤價第一次跌破下軌線', 
             xy=(pd.Timestamp('2008-6-30'), taiex_close.loc['2008-6-30']),  # 箭頭指向的點
             xytext=(pd.Timestamp('2008-6-30'), taiex_close.loc['2008-6-30'] + 500),  # 文字位置往上移
             arrowprops=dict(
                arrowstyle='->',
                connectionstyle='angle,angleA=90,angleB=0',  # 使箭頭垂直
                shrinkA=0,  
                shrinkB=0   
             ),
             fontsize=12, 
             color='red',
             ha='center',  # 水平置中對齊
             va='bottom'   # 垂直對齊在文字底部
)

plt.annotate('收盤價再度跌破下軌線，\n且價格低於第一次跌破價格', 
             xy=(pd.Timestamp('2008-7-25'), taiex_close.loc['2008-7-25']),
             xytext=(pd.Timestamp('2008-7-25'), taiex_close.loc['2008-7-25'] - 2000),
             arrowprops=dict(
                arrowstyle='->',
                connectionstyle='angle,angleA=90,angleB=0',
                shrinkA=0,  
                shrinkB=0   
             ),
             fontsize=14, 
             color='red',
             ha='center',
             va='bottom'
)

plt.annotate('收盤價回到均線之上', 
             xy=(pd.Timestamp('2009-5-5'), taiex_close.loc['2009-5-5']),
             xytext=(pd.Timestamp('2009-5-5'), taiex_close.loc['2009-5-5'] + 1000),
             arrowprops=dict(
                arrowstyle='->',
                connectionstyle='angle,angleA=90,angleB=0',
                shrinkA=0,  
                shrinkB=0   
             ),
             fontsize=14, 
             color='red',
             ha='center',
             va='bottom'
)

plt.title("台股大盤布林通道（MA300 標準差=2）", fontsize=16)
plt.xlabel("年份", fontsize=14)
plt.xticks(fontsize=14)
plt.ylabel("台股大盤")
plt.yticks(fontsize=14)

plt.legend(fontsize=14)
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

### 建立布林通道濾網交易訊號

In [ ]:
bolling_filt = orig_all_cond.copy()
aligned_signal = tw_bollinger_signal.reindex(bolling_filt.index, method='ffill', fill_value=True)
bolling_filt.loc[aligned_signal.index, :] = aligned_signal.values[:, None]

### 有無濾網_綜合比較

In [ ]:
bollinger_compare_dict = {}

# END_DATE = '2009-3-31'
# END_DATE = '2024-12-31'

bollinger_compare_dict['原始條件_無本益比'] = orig_all_cond[START_DATE:END_DATE]
bollinger_compare_dict['原始條件_無本益比_ROE出場條件'] = orig_all_cond[START_DATE:END_DATE] & (roe[START_DATE:END_DATE] > 15)
bollinger_compare_dict['原始條件_無本益比_布林通道'] = orig_all_cond[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]
bollinger_compare_dict['原始條件_無本益比_ROE出場條件_布林通道'] = orig_all_cond[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE] & (roe[START_DATE:END_DATE] > 15)

bollinger_compare_dict['原始條件_有本益比'] = orig_all_cond_and_pe_daily[START_DATE:END_DATE]
bollinger_compare_dict['原始條件_有本益比_ROE出場條件'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 15))
bollinger_compare_dict['原始條件_有本益比_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])
bollinger_compare_dict['原始條件_有本益比_ROE出場條件_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 15)) 

In [ ]:
bollinger_compare_collecs = sim_conditions(bollinger_compare_dict, resample='M', data=data)

In [ ]:
bollinger_compare_collecs.plot_strategies_cumm_return()

In [ ]:
bollinger_compare_collecs.plot_strategies_MDD()

In [ ]:
bollinger_compare_collecs.selected_stock_count_analysis()

In [ ]:
bollinger_compare_collecs.reports['原始條件_有本益比_布林通道'].display()

In [ ]:
# bollinger_compare_collecs.reports['原始條件_有本益比_布林通道'].trades.to_csv('台股原始條件_有本益比_布林通道.csv', encoding='utf-8-sig')

In [ ]:
bollinger_compare_collecs.plot_reps_stock_counts()

In [ ]:
bollinger_compare_df = bollinger_compare_collecs.selected_stock_count_analysis().reset_index()
# 提取策略前綴，移除 '_布林通道' 作為分組依據
bollinger_compare_df["Base_Strategy"] = bollinger_compare_df["Strategy"].str.replace("_布林通道", "", regex=False)
bollinger_compare_df["Condition"] = bollinger_compare_df["Strategy"].str.contains("_布林通道")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))

# CAGR
cagr_comparison = bollinger_compare_df.pivot_table(
    index="Base_Strategy",
    columns="Condition",
    values="CAGR (%)"
)
cagr_comparison.columns = ["無布林通道", "布林通道"]
cagr_comparison = cagr_comparison.reset_index()

cagr_comparison.plot(
    x="Base_Strategy",
    kind="bar",
    ax=ax1,
    xlabel="策略",
    rot=45,
    color=['tab:blue', 'orange'],
    fontsize=16,
    legend=False  # 移除個別的legend
)
ax1.set_title(f"台股 2003~{END_DATE} 布林通道 vs 無布林通道的 CAGR (%) 比較", fontsize=16)
ax1.set_ylabel("CAGR (%)", fontsize=12)
ax1.grid(axis='y', alpha=0.4, linestyle='--')

# MDD 
mdd_comparison = bollinger_compare_df.pivot_table(
    index="Base_Strategy",
    columns="Condition",
    values="MDD (%)"
)
mdd_comparison.columns = ["無布林通道", "布林通道"]
mdd_comparison = mdd_comparison.reset_index()

mdd_comparison.plot(
    x="Base_Strategy",
    kind="bar",
    ax=ax2,
    xlabel="策略",
    rot=45,
    color=['tab:blue', 'orange'],
    fontsize=16,
    legend=False
)
ax2.set_title(f"台股 2003~{END_DATE} 布林通道 vs 無布林通道的 MDD (%) 比較", fontsize=16)
ax2.set_ylabel("MDD (%)", fontsize=12)
ax2.grid(axis='y', alpha=0.4, linestyle='--')

# 添加共同的legend在底部中央
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, 
          loc='center',
          bbox_to_anchor=(0.5, -0.01),  # 調整legend的位置
          ncol=2,  # 將legend排成兩列
          fontsize=12,
          title_fontsize=12)

plt.tight_layout()
plt.show()

# fig.savefig('./img/圖 19台股2003年至2024年有無加上濾網策略績效比較圖.svg', dpi=300, format='svg', bbox_inches='tight')

---

## 調整財務比率篩選標準

### 進場條件
- 本益比進出場

In [ ]:
pe_entry_test_dic = {}

for p in range(8, 15, 2):
    pe_entry_test = daily_pe.resample('M').last() < p
    pe_entry_test_dic[f'原始條件_本益比小於{p}進場'] = (orig_all_cond & pe_entry_test[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE])

pe_entry_test_comb = sim_conditions(pe_entry_test_dic, resample="M", data=data)
pe_entry_test_comb.selected_stock_count_analysis()

---

## 兩兩一組

In [ ]:
def plot_strategy_heatmap(df, compare='CAGR (%)', x_first=True, figsize=(14, 6), title=None, 
                         benchmark_param1=None, benchmark_param2=None, rep=None):
    """
    繪製策略熱力圖，當df['Min']==0時顯示灰色，並用白色框線標記Benchmark位置
    如果策略的日期不是從2003年開始，也會顯示灰色

    Parameters:
    -----------
    df : pandas DataFrame
        包含 'Strategy' 和比較欄位的數據框
    compare : str, default='CAGR (%)'
        要比較的欄位名稱，例如 'CAGR (%)' 或 'MDD (%)'
    x_first : bool, default=True
        True: 條件1為X軸，條件2為Y軸
        False: 條件1為Y軸，條件2為X軸
    figsize : tuple, default=(14, 6)
        圖形尺寸
    title : str, optional
        圖表標題，如果不指定則自動生成
    benchmark_param1 : float, optional
        指標1的基準值
    benchmark_param2 : float, optional
        指標2的基準值
    """
    
    if compare not in df.columns:
        raise ValueError(f"Column '{compare}' not found in DataFrame")
    
    param1_values = []
    param2_values = []
    not_start_2003 = []  # 儲存不是從2003年開始的策略
    
    # 從策略名稱中提取參數
    for strategy in df['Strategy']:
        date_index = rep.reports[strategy].position.index
        # print(date_index)
        
        # 檢查是否從2003年開始
        starts_from_2003 = False
        if len(date_index) > 0:
            first_date_str = str(date_index[0])
            if first_date_str.startswith('2003'):
                starts_from_2003 = True
                
        parts = strategy.split('_')
        # 處理可能帶有%的數值
        param1 = float(parts[1].replace('%', ''))
        param2 = float(parts[3].replace('%', ''))
        param1_values.append(param1)
        param2_values.append(param2)
        not_start_2003.append(not starts_from_2003)  # 記錄非2003開始的策略
    
    df['Param1'] = param1_values
    df['Param2'] = param2_values
    df['Not2003Start'] = not_start_2003  # 將結果添加到DataFrame
    
    condition1_name = df['Strategy'].iloc[0].split('_')[0]
    condition2_name = df['Strategy'].iloc[0].split('_')[2]

    if x_first:
        pivot_table = df.pivot(
            index='Param2',
            columns='Param1',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition1_name} (%)"
        ylabel = f"{condition2_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param1)
            benchmark_y = pivot_table.index.get_loc(benchmark_param2)
    else:
        pivot_table = df.pivot(
            index='Param1',
            columns='Param2',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition2_name} (%)"
        ylabel = f"{condition1_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param2)
            benchmark_y = pivot_table.index.get_loc(benchmark_param1)
    
    plt.figure(figsize=figsize)
    
    mask = (min_pivot == 0)
    
    # 繪製熱力圖
    sns.heatmap(pivot_table,
                annot=True,
                annot_kws={'size': 14},
                fmt='.2f',
                cmap='coolwarm',
                cbar_kws={'label': compare},
                square=True,
                mask=None)
    
    # 在Min==0或不是從2003年開始的位置上覆蓋統一的灰色方塊
    for i in range(len(pivot_table.index)):
        for j in range(len(pivot_table.columns)):
            # 如果Min==0或不是從2003年開始，則繪製灰色方塊
            if mask.iloc[i, j] or (not_2003_pivot.iloc[i, j] == True):
                plt.gca().add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='#808080'))
    
    # 如果有指定Benchmark參數，繪製白色框線
    if benchmark_param1 is not None and benchmark_param2 is not None:
        plt.gca().add_patch(plt.Rectangle((benchmark_x, benchmark_y), 1, 1, 
                                        fill=False, edgecolor='white', linewidth=2))
    
    if title is None:
        title = f'{condition1_name} vs {condition2_name} 策略{compare}比較'
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()

### ROE & 盈再率

In [ ]:
roe_rr_opt_base_cond = payout_ratio_cond & profit_cond & hold_cond & listed

roe_rr_pe_opt_conds = {}

for r in range(0, 81, 10): # 小於 0~40% (80%)
    for p in range(10, 26, 5): # ROE 10~25%
        roe_5y_opt = roe.rolling(5).mean() > p
        rr_opt = rr < (r/100)

        roe_rr_opt_all_conds = (roe_5y_opt & rr_opt & roe_rr_opt_base_cond)[START_DATE:END_DATE]
        roe_rr_pe_opt_conds[f'ROE5年平均_{p}_盈再率_{r}__本益比進出場'] = (roe_rr_opt_all_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until(~roe_rr_opt_all_conds | pe_cond_exit_daily[START_DATE:END_DATE])

roe_rr_pe_opt_comb = sim_conditions(roe_rr_pe_opt_conds, resample='M', data=data)
roe_rr_pe_opt_comb.selected_stock_count_analysis()

In [ ]:
roe_rr_pe_opt_comb.plot_reps_stock_counts(['ROE5年平均_10_盈再率_0__本益比進出場', 'ROE5年平均_15_盈再率_0__本益比進出場', 'ROE5年平均_15_盈再率_40__本益比進出場'])

In [ ]:
roe_rr_opt_df = roe_rr_pe_opt_comb.selected_stock_count_analysis()
roe_rr_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_rr_opt_df,
                      x_first=False,
                         title='2003~2024 台股_ROE5年平均_盈再率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40, rep = roe_rr_pe_opt_comb)

In [ ]:
plot_strategy_heatmap(roe_rr_opt_df,
                    compare='MDD (%)',
                    x_first=False,
                    title='2003~2024 台股_ROE5年平均_盈再率_本益比進出場_MDD(%)',
                    figsize=(16, 6),
                    benchmark_param1=15,
                    benchmark_param2=40, rep = roe_rr_pe_opt_comb)

### ROE & 本益比

In [ ]:
roe_pee_opt_base_conds = rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed

roe_pee_opt_conds = {}

for roevalue in range(10, 26, 5): # ROE 10~25%
    for peevalue in range(8, 13, 2):  # 本益比

        roe_5y_opt = roe.rolling(5).mean() > roevalue
        pee_opt = (daily_pe < peevalue).resample('M').last()
        
        roe_pee_opt_all_conds = (roe_pee_opt_base_conds & roe_5y_opt)[START_DATE:END_DATE]

        roe_pee_opt_conds[f'ROE5年平均_{roevalue}%_本益比_{peevalue}__本益比進出場'] = (roe_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~roe_pee_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])


roe_pee_opt_collecs = sim_conditions(roe_pee_opt_conds, resample='M', data=data)
roe_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_pe_opt_df = roe_pee_opt_collecs.selected_stock_count_analysis()
roe_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_pe_opt_df,
                    title='2003~2024 台股_ROE5年平均_本益比進出場_CAGR(%)',
                    figsize=(18, 6),
                    benchmark_param1=15,
                    benchmark_param2=12,
                    rep = roe_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_pe_opt_df,
                    compare='MDD (%)',
                    title='2003~2024 台股_ROE5年平均_本益比進出場_MDD(%)',
                    figsize=(18, 6),
                    benchmark_param1=15,
                    benchmark_param2=12,
                    rep = roe_pee_opt_collecs)

### ROE & 股利支付率

In [ ]:
roe_dpr_opt_base_conds = rr_cond & profit_cond & hold_cond & listed

roe_dpr_pe_opt_conds = {}

for i in range(10, 26, 5): # ROE 10~25%
    for n in range(0, 56, 5):  # 配息率 0~55%
        dpr_cond_rr_opt = payout_ratio.rolling(3).min() > n
        roe_5y_opt = roe.rolling(5).mean() > i
        
        roe_dpr_opt_all_conds = (roe_dpr_opt_base_conds & roe_5y_opt & dpr_cond_rr_opt)[START_DATE:END_DATE]

        roe_dpr_pe_opt_conds[f'ROE5年平均_{i}_配息率_{n}__本益比進出場'] = (roe_dpr_opt_all_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~roe_dpr_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])



roe_dpr_pe_opt_collecs = sim_conditions(roe_dpr_pe_opt_conds, resample='M', data=data)
roe_dpr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_dpr_pe_opt_df = roe_dpr_pe_opt_collecs.selected_stock_count_analysis()
roe_dpr_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_dpr_pe_opt_df,
                        x_first=False, 
                        title='2003~2024 台股_ROE5年平均_配息率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_dpr_pe_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_dpr_pe_opt_df,
                        x_first=False,
                        compare='MDD (%)',
                        title='2003~2024 台股_ROE5年平均_配息率_本益比進出場_MDD(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_dpr_pe_opt_collecs)

### 盈再率 & 本益比

In [ ]:
rr_pee_opt_base_conds = roe_15 & payout_ratio_cond & profit_cond & hold_cond & listed

rr_pee_opt_conds = {}

for rrvalue in range(0, 81, 10): # 盈再率
    for peevalue in range(8, 13, 2):  # 本益比

        pee_opt = (daily_pe < peevalue).resample('M').last()
        rrvalue_opt = rr.copy() < (rrvalue/100)
        
        rr_pee_opt_all_conds = (rr_pee_opt_base_conds & rrvalue_opt)[START_DATE:END_DATE]

        rr_pee_opt_conds[f'盈再率小於_{rrvalue}%_本益比_{peevalue}__本益比進出場'] = (rr_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~rr_pee_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])

rr_pee_collecs = sim_conditions(rr_pee_opt_conds, resample='M', data=data)
rr_pee_collecs.selected_stock_count_analysis()

In [ ]:
rr_pee_collecs.plot_reps_stock_counts(["盈再率小於_0%_本益比_12__本益比進出場", "盈再率小於_40%_本益比_12__本益比進出場"])

In [ ]:
rr_pee_opt_df = rr_pee_collecs.selected_stock_count_analysis()
rr_pee_opt_df.reset_index(inplace=True)

plot_strategy_heatmap(rr_pee_opt_df,
                        x_first=True, 
                        title='2003~2024 台股_盈再率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = rr_pee_collecs)

In [ ]:
plot_strategy_heatmap(rr_pee_opt_df,
                        x_first=True, 
                        compare='MDD (%)',
                        title='2003~2024 台股_盈再率_本益比進出場_MDD(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = rr_pee_collecs)

### 股利支付率 & 本益比

In [ ]:
dpr_pee_opt_base_conds = roe_15 & rr_cond & profit_cond & hold_cond & listed

dpr_pee_opt_conds = {}

for povalue in range(0, 56, 5): # 配息率
    for peevalue in range(8, 13, 2):  # 本益比

        pee_opt = (daily_pe < peevalue).resample('M').last()
        dpr_cond_rr_opt = payout_ratio.rolling(3).min() >= povalue
        
        dpr_pee_opt_all_conds = (dpr_pee_opt_base_conds & dpr_cond_rr_opt)[START_DATE:END_DATE]

        dpr_pee_opt_conds[f'配息率_{povalue}%_本益比_{peevalue}__本益比進出場'] = (dpr_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~dpr_pee_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])

dpr_pee_opt_collecs = sim_conditions(dpr_pee_opt_conds, resample='M', data=data)
dpr_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
dpr_pee_opt_df = dpr_pee_opt_collecs.selected_stock_count_analysis()
dpr_pee_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(dpr_pee_opt_df,
                        x_first=True, 
                        compare='CAGR (%)',
                        title='2003~2024 台股_配息率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = dpr_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(dpr_pee_opt_df,
                        x_first=True, 
                        compare='MDD (%)',
                        title='2003~2024 台股_配息率_本益比進出場_MDD(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = dpr_pee_opt_collecs)